In [1]:
import pandas as pd
import google.generativeai as genai
import json
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

C:\Users\USER\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\USER\AppData\Local\Temp\ipykernel_12508\4093282566.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
def prep_data_for_llm(file_path):
    print("Memuat data mutasi...")
    # 1. Load data CSV
    df = pd.read_csv(file_path)
    
    # 2. Filter data: Kita hanya peduli pada Pengeluaran (Debit)
    # Pemasukan (Gaji) tidak perlu dikategorikan oleh AI
    df_expenses = df[df['Type'] == 'Debit'].copy()
    
    # 3. Formatting Tipe Data
    # Ubah kolom Date menjadi format datetime standar
    df_expenses['Date'] = pd.to_datetime(df_expenses['Date'])
    
    # Pastikan Amount terbaca sebagai angka (integer)
    df_expenses['Amount'] = pd.to_numeric(df_expenses['Amount'])
    
    # 4. Cleaning & Merapikan Kolom
    # Karena semua sisa data adalah Debit, kolom 'Type' sudah tidak relevan, kita drop saja
    df_expenses = df_expenses.drop(columns=['Type'])
    
    # Reset index agar urutannya kembali rapi dari 0
    df_expenses = df_expenses.reset_index(drop=True)
    
    return df_expenses

# Menjalankan fungsi
if __name__ == "__main__":
    file_name = 'dummy_transactions.csv'
    
    # Panggil fungsi pembersihan
    cleaned_df = prep_data_for_llm(file_name)
    
    print("\nData Pengeluaran (Siap dikirim ke LLM):")
    print(cleaned_df)
    
    print(f"\nTotal transaksi pengeluaran yang akan diproses: {len(cleaned_df)} baris")

Memuat data mutasi...

Data Pengeluaran (Siap dikirim ke LLM):
         Date  Amount             Raw_Description
0  2026-08-02   25000              QRIS KOPI HOJA
1  2026-08-03  150000           PAYMENT PLN TOKEN
2  2026-08-04   18000             TRF KANTIN RAYA
3  2026-08-05   54900             SPOTIFY PREMIUM
4  2026-08-07  350000  TIKET KERETA KAI KAI-8921X
5  2026-08-08   22000      QRIS WAROENK PENGKOLAN
6  2026-08-10  150000                 TOPUP GOPAY
7  2026-08-12   45000           GRABUNLIMITED SUB
8  2026-08-15   12500             TRF BORJO MURNI
9  2026-08-16  320000         SHOPEE PAY INV-9912
10 2026-08-18   30000         QRIS KANCANE COFFEE
11 2026-08-20   75000            EVERYDAY LAUNDRY
12 2026-08-22  120000              ORANGE CARWASH
13 2026-08-25   50000              WD ATM MANDIRI

Total transaksi pengeluaran yang akan diproses: 14 baris


In [3]:
# 1. Konfigurasi API
load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=API_KEY, transport='rest')

# Menggunakan model gemini-3.5-flash-lite
model = genai.GenerativeModel('gemini-3.5-flash-lite')

print("Mulai proses kategorisasi dengan AI (Batch Mode)...\n")

# 2. Ambil seluruh list deskripsi transaksi
descriptions = cleaned_df['Raw_Description'].tolist()

# 3. Kirim semuanya dalam 1 Prompt Tunggal (Batching)
prompt = f"""
Kamu adalah asisten keuangan otomatis. Kategorikan setiap nama transaksi berikut ke SALAH SATU dari kategori ini:
[F&B, Transportasi, Utilitas, Hiburan, Belanja, Perawatan Kendaraan, Transfer/Tarik Tunai, Lainnya]

Daftar Transaksi:
{json.dumps(descriptions, ensure_ascii=False)}

ATURAN SANGAT PENTING:
Keluarkan HANYA JSON array string berisi nama kategorinya saja dengan urutan yang sama persis.
Contoh format output:
["F&B", "Utilitas", "Hiburan"]
"""

try:
    response = model.generate_content(prompt)
    clean_json_text = response.text.strip().replace("```json", "").replace("```", "").strip()
    ai_categories = json.loads(clean_json_text)
    
    # Masukkan hasil AI ke dalam DataFrame
    cleaned_df['AI_Category'] = ai_categories
    print(f"✅ Berhasil mengkategorikan {len(ai_categories)} transaksi sekaligus!")
    for desc, cat in zip(descriptions, ai_categories):
        print(f"   [{cat}] <- {desc}")
except Exception as e:
    print(f"Error saat kategorisasi: {e}")


Mulai proses kategorisasi dengan AI (Batch Mode)...

✅ Berhasil mengkategorikan 14 transaksi sekaligus!
   [F&B] <- QRIS KOPI HOJA
   [Utilitas] <- PAYMENT PLN TOKEN
   [F&B] <- TRF KANTIN RAYA
   [Hiburan] <- SPOTIFY PREMIUM
   [Transportasi] <- TIKET KERETA KAI KAI-8921X
   [F&B] <- QRIS WAROENK PENGKOLAN
   [Transportasi] <- TOPUP GOPAY
   [Hiburan] <- GRABUNLIMITED SUB
   [Belanja] <- TRF BORJO MURNI
   [Belanja] <- SHOPEE PAY INV-9912
   [F&B] <- QRIS KANCANE COFFEE
   [Utilitas] <- EVERYDAY LAUNDRY
   [Perawatan Kendaraan] <- ORANGE CARWASH
   [Transfer/Tarik Tunai] <- WD ATM MANDIRI


In [4]:
# Tampilkan tabel akhirnya
cleaned_df

,Date,Amount,Raw_Description,AI_Category
0,2026-08-02,25000,QRIS KOPI HOJA,F&B
1,2026-08-03,150000,PAYMENT PLN TOKEN,Utilitas
2,2026-08-04,18000,TRF KANTIN RAYA,F&B
3,2026-08-05,54900,SPOTIFY PREMIUM,Hiburan
4,2026-08-07,350000,TIKET KERETA KAI KAI-8921X,Transportasi
5,2026-08-08,22000,QRIS WAROENK PENGKOLAN,F&B
6,2026-08-10,150000,TOPUP GOPAY,Transportasi
7,2026-08-12,45000,GRABUNLIMITED SUB,Hiburan
8,2026-08-15,12500,TRF BORJO MURNI,Belanja
9,2026-08-16,320000,SHOPEE PAY INV-9912,Belanja


In [5]:
# 1. Memuat variabel rahasia dari file .env
load_dotenv()

USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# 2. Merakit string koneksi
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

print("Mencoba terhubung ke Supabase...")
engine = create_engine(DATABASE_URL)

try:
    # 3. Mendorong DataFrame ke tabel 'expenses' di PostgreSQL
    # Pastikan variabel cleaned_df dari proses AI sebelumnya masih ada di memori Jupyter lu
    cleaned_df.to_sql('expenses', engine, if_exists='replace', index=False)
    
    print("✅ Berhasil! Data pengeluaran dan kategori AI sudah tersimpan di Supabase.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat mengirim data: {e}")

Mencoba terhubung ke Supabase...


✅ Berhasil! Data pengeluaran dan kategori AI sudah tersimpan di Supabase.
